# "FINE TUNING" Project with Gradio

Fine Tune different frontier models, that predicts how much something costs from a description, based on a scrape of Amazon data, and compare the prices
 
Now we will use followig models to fine-tune
1. OpenAI - GPT-4.1-nano - using supervised fine tuning
2. Anthropic - claude-haiku-4-5-20251001 & claude-sonnet-4-6 (using few shot prompting) as they do not have 
   fine tuning


In [ ]:
import os
import re
import json
import time
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

from typing import Any, Callable, Dict, List, Optional
from typing import List, Dict, Optional
from litellm import completion
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [ ]:
# Load API Keys and 

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
hf_token = os.environ['HF_TOKEN']


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")

if hf_token:
    print(f"hf_token exists and begins {hf_token[:3]}")
    login(hf_token, add_to_git_credential=True)
else:
    print("hf_token is not set (and this is optional)")

In [ ]:
# Load only 20,000 price data

LITE_MODE = True

In [ ]:
username = "tezkrishna"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# set the clients
openai = OpenAI()
anthropic_url = "https://api.anthropic.com/v1/"
#gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
#gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [ ]:
len(fine_tune_train)

In [ ]:
SYSTEM_MESSAGE = f""" You are a product price prediction model.

Given a product summary, estimate the most likely selling price.

Strict output rules:
- Return only one price
- Format: $123.45
- No explanation
- No additional text
- No commas unless required for thousands, for example: $1,249.00
- Never return a range
- Never return multiple prices
- Never return "unknown" or "N/A"
- Always produce a single best estimate
"""

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
messages_for(fine_tune_train[0])

In [ ]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
print(make_jsonl(train[:3]))

In [ ]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
train_file

In [ ]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
validation_file

# Open AI Fine Tuning

In [ ]:
# Fine Tuner class
class OpenAIFineTuner:
    TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}

    def __init__(
        self,
        api_key: Optional[str] = None,
        client: Optional[OpenAI] = None,
    ):
        self.client = client or OpenAI(
            api_key=api_key or os.environ.get("OPENAI_API_KEY")
        )

    def create_job(
        self,
        train_file_id: str,
        validation_file_id: Optional[str] = None,
        model: str = "gpt-4.1-nano-2025-04-14",
        seed: int = 42,
        n_epochs: int = 1,
        batch_size: int = 1,
        suffix: str = "price-estimator",
    ):
        """
        Create a supervised fine-tuning job.
        """
        payload: Dict[str, Any] = {
            "training_file": train_file_id,
            "model": model,
            "seed": seed,
            "suffix": suffix,
            "method": {
                "type": "supervised",
                "supervised": {
                    "hyperparameters": {
                        "n_epochs": n_epochs,
                        "batch_size": batch_size,
                    }
                },
            },
        }

        if validation_file_id:
            payload["validation_file"] = validation_file_id

        return self.client.fine_tuning.jobs.create(**payload)

    def list_jobs(self, limit: int = 10):
        return self.client.fine_tuning.jobs.list(limit=limit)

    def get_latest_job(self):
        jobs = self.client.fine_tuning.jobs.list(limit=1)
        if not jobs.data:
            raise ValueError("No fine-tuning jobs found.")
        return jobs.data[0]

    def get_latest_job_id(self) -> str:
        return self.get_latest_job().id

    def retrieve_job(self, job_id: str):
        return self.client.fine_tuning.jobs.retrieve(job_id)

    def get_job_status(self, job_id: str) -> str:
        job = self.retrieve_job(job_id)
        return job.status

    def list_events(self, job_id: str, limit: int = 10):
        return self.client.fine_tuning.jobs.list_events(
            fine_tuning_job_id=job_id,
            limit=limit,
        ).data

    def get_fine_tuned_model_name(self, job_id: str) -> Optional[str]:
        job = self.retrieve_job(job_id)
        return job.fine_tuned_model

    def wait_for_completion(
        self,
        job_id: str,
        poll_interval: int = 30,
        print_progress: bool = True,
        include_events: bool = False,
        events_limit: int = 10,
        on_update: Optional[Callable[[str, Dict[str, Any]], None]] = None,
    ):
        """
        Poll until the fine-tuning job finishes.

        on_update callback signature:
            on_update(message: str, payload: dict) -> None
        """
        seen_event_ids = set()

        while True:
            job = self.retrieve_job(job_id)
            status = job.status

            update_payload = {
                "job_id": job_id,
                "status": status,
                "fine_tuned_model": getattr(job, "fine_tuned_model", None),
                "estimated_finish": getattr(job, "estimated_finish", None),
                "trained_tokens": getattr(job, "trained_tokens", None),
                "error": getattr(job, "error", None),
            }

            message = f"Fine-tuning job {job_id} status: {status}"

            if print_progress:
                print(message)

            if on_update:
                on_update(message, update_payload)

            if include_events:
                try:
                    events = self.list_events(job_id, limit=events_limit)
                    for event in reversed(events):
                        event_id = getattr(event, "id", None)
                        if event_id and event_id in seen_event_ids:
                            continue

                        event_msg = f"Event: {event}"
                        if print_progress:
                            print(event_msg)
                        if on_update:
                            on_update(
                                event_msg,
                                {
                                    "job_id": job_id,
                                    "event": event,
                                },
                            )

                        if event_id:
                            seen_event_ids.add(event_id)
                except Exception as e:
                    warn_msg = f"Could not fetch events for job {job_id}: {e}"
                    if print_progress:
                        print(warn_msg)
                    if on_update:
                        on_update(
                            warn_msg,
                            {
                                "job_id": job_id,
                                "warning": str(e),
                            },
                        )

            if status in self.TERMINAL_STATUSES:
                if status == "succeeded":
                    done_msg = (
                        f"Fine-tuning finished successfully. "
                        f"Fine-tuned model: {job.fine_tuned_model}"
                    )
                elif status == "failed":
                    done_msg = f"Fine-tuning failed. Error: {job.error}"
                else:
                    done_msg = "Fine-tuning was cancelled."

                if print_progress:
                    print(done_msg)

                if on_update:
                    on_update(
                        done_msg,
                        {
                            "job_id": job_id,
                            "status": status,
                            "fine_tuned_model": getattr(job, "fine_tuned_model", None),
                            "error": getattr(job, "error", None),
                        },
                    )

                return job

            time.sleep(poll_interval)

    def create_and_track(
        self,
        train_file_id: str,
        validation_file_id: Optional[str] = None,
        model: str = "gpt-4.1-nano-2025-04-14",
        seed: int = 42,
        n_epochs: int = 1,
        batch_size: int = 1,
        suffix: str = "price-estimator",
        events_limit: int = 10,
        wait: bool = False,
        poll_interval: int = 30,
        print_progress: bool = True,
        include_events: bool = True,
        on_update: Optional[Callable[[str, Dict[str, Any]], None]] = None,
    ) -> Dict[str, Any]:
        """
        Create job and optionally wait until it finishes.
        """
        created_job = self.create_job(
            train_file_id=train_file_id,
            validation_file_id=validation_file_id,
            model=model,
            seed=seed,
            n_epochs=n_epochs,
            batch_size=batch_size,
            suffix=suffix,
        )

        job_id = created_job.id

        if print_progress:
            print(f"Created fine-tuning job: {job_id}")

        if on_update:
            on_update(
                f"Created fine-tuning job: {job_id}",
                {"job_id": job_id, "created_job": created_job},
            )

        retrieved_job = self.retrieve_job(job_id)
        events = self.list_events(job_id, limit=events_limit)
        fine_tuned_model_name = retrieved_job.fine_tuned_model

        result = {
            "job_id": job_id,
            "created_job": created_job,
            "retrieved_job": retrieved_job,
            "events": events,
            "fine_tuned_model_name": fine_tuned_model_name,
        }

        if wait:
            final_job = self.wait_for_completion(
                job_id=job_id,
                poll_interval=poll_interval,
                print_progress=print_progress,
                include_events=include_events,
                events_limit=events_limit,
                on_update=on_update,
            )
            result["final_job"] = final_job
            result["fine_tuned_model_name"] = final_job.fine_tuned_model

        return result

In [ ]:
# create the class and call the function to create the job

fine_tuner = OpenAIFineTuner()

result = fine_tuner.create_and_track(
    train_file_id=train_file.id,
    validation_file_id=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    n_epochs=1,
    batch_size=1,
    suffix="price-estimator",
    wait=False,
    poll_interval=120,
    include_events=True,
)

print("Job ID:", result["job_id"])
print("Events:", result["events"])
print("Fine-tuned model:", result.get("fine_tuned_model_name"))

In [ ]:
# list the job
openai.fine_tuning.jobs.list(limit=1)

In [ ]:
# get the jobid
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
job_id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
# retrieve job id and lost events
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

In [ ]:
jobs = openai.fine_tuning.jobs.list(limit=10)
for j in jobs.data:
    print(j.id, j.status)

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
def predict_openai(summary: str, model_name: str = fine_tuned_model_name) -> str:
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        temperature=0,
        max_tokens=20,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": f"Estimate price:\n\n{summary}"}
        ]
    )
    return response.choices[0].message.content.strip()

In [ ]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": message},
    ]

In [ ]:
test[0]

In [ ]:

def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

# Anthropic - Fine Tunning 
## It is through few shot prompting 

In [ ]:
print(fine_tune_train)

In [ ]:
#method to build anthropic message
def build_anthropic_messages(summary: str, fewshot_items=None, max_examples: int = 10):
    messages = []

    
    if fewshot_items is None:
        fewshot_items = fine_tune_train

    #print(fewshot_items)
    if fewshot_items:
        for item in fewshot_items[:max_examples]:
            messages.append({
                "role": "user",
                "content": f"Estimate price:\n\n{item.summary}"
            })
            messages.append({
                "role": "assistant",
                "content": f"${item.price:.2f}"
            })

    messages.append({
        "role": "user",
        "content": f"Estimate price:\n\n{summary}"
    })
    #print(messages)
    return messages

In [ ]:
from anthropic import Anthropic
anthropic = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [ ]:
# method to predic the price
def predict_price_anthropic(summary: str, model_name: str = "claude-haiku-4-5-20251001", fewshot_items=None) -> str:
    response = anthropic.messages.create(
        model=model_name,
        system=SYSTEM_MESSAGE,
        temperature=0,
        max_tokens=10,
        messages= build_anthropic_messages(summary, fewshot_items)
    )
 
    output = []
    for block in response.content:
        if getattr(block, "type", None) == "text":
            output.append(block.text)

    return "".join(output).strip()

In [ ]:
# helpers
def parse_price(text: str) -> float:
    if text is None:
        raise ValueError("Prediction text is None")

    match = re.search(r"\d+(?:,\d{3})*(?:\.\d+)?", str(text))
    if not match:
        raise ValueError(f"Could not parse price from: {text}")

    return float(match.group(0).replace(",", ""))

In [ ]:
from dataclasses import dataclass

@dataclass
class Item:
    summary: str
    price: float

In [ ]:
def parse_fewshot_text(fewshot_text: str) -> List[Item]:
    """
    Format:
    summary || price
    summary || price
    """
    items = []
    if not fewshot_text.strip():
        return items
        
    for line in fewshot_text.strip().splitlines():
        if "||" not in line:
            continue
        print(line)
        summary, price = line.split("||", 1)
        items.append(Item(summary=summary.strip(), price=float(price.strip())))
    return items

In [ ]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": message},
    ]

In [ ]:
# Try this out

test_messages_for(test[0])

In [ ]:
# The inference function

def claude_few_shot_fine_tuned(item):
    response = predict_price_anthropic(item.summary)
    return response

In [ ]:
print(test[0].summary)
type(test[0].summary)

In [ ]:
print(test[0].price)
print(claude_few_shot_fine_tuned(test[0]))

In [ ]:
evaluate(claude_few_shot_fine_tuned, test)

In [ ]:
def build_bar_chart(plot_df: pd.DataFrame):
    if plot_df is None or plot_df.empty:
        return None

    fig, ax = plt.subplots(figsize=(6, 4), facecolor="white")
    ax.set_facecolor("white")

    labels = plot_df["label"].tolist()
    values = plot_df["predicted_price"].tolist()

    bars = ax.bar(labels, values, width=0.25)  # smaller width

    ax.set_ylabel("Predicted Price")
    ax.set_title("Predicted Price by Model")
    ax.set_ylim(0, max(values) * 1.2)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${value:.2f}",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    return fig

In [ ]:
# Models list
OPENAI_MODELS = ["gpt-4.1-nano"]
ANTHROPIC_MODEL_OPTIONS = {
    "Claude Haiku 4.5 (cheaper)": "claude-haiku-4-5-20251001",
    "Claude Sonnet 4.6": "claude-sonnet-4-6",
}

In [ ]:
#UI function

def run_price_estimation(
    summary: str,
    mode: str,
    openai_model_name: str,
    anthropic_model_label: str,
    fewshot_text: str
):

    if not summary.strip():
        raise ValueError("Please enter a product summary.")

    fewshot_items = parse_fewshot_text(fewshot_text)
    anthropic_model_name = ANTHROPIC_MODEL_OPTIONS[anthropic_model_label]
    print(fewshot_items)
    rows = []

    if mode == "OpenAI only":
        raw = predict_openai(summary, openai_model_name.strip())
        pred = parse_price(raw)
        rows.append({
            "provider": "OpenAI",
            "model_name": openai_model_name.strip(),
            "raw_output": raw,
            "predicted_price": pred
        })

    elif mode == "Anthropic only":
        raw = predict_price_anthropic(
            summary=summary,
            model_name=anthropic_model_name.strip(),
            fewshot_items=fewshot_items
        )
        pred = parse_price(raw)
        rows.append({
            "provider": "Anthropic",
            "model_name": anthropic_model_name.strip(),
            "raw_output": raw,
            "predicted_price": pred
        })

    elif mode == "Both":
        # OpenAI
        try:
            raw_openai = predict_openai(summary, openai_model_name.strip())
            pred_openai = parse_price(raw_openai)
            rows.append({
                "provider": "OpenAI",
                "model_name": openai_model_name.strip(),
                "raw_output": raw_openai,
                "predicted_price": pred_openai
            })
        except Exception as e:
            rows.append({
                "provider": "OpenAI",
                "model_name": openai_model_name.strip(),
                "raw_output": f"ERROR: {e}",
                "predicted_price": None
            })

        # Anthropic
        try:
            raw_anthropic = predict_price_anthropic(
                summary=summary,
                model_name=anthropic_model_name.strip(),
                fewshot_items=fewshot_items
            )
            pred_anthropic = parse_price(raw_anthropic)
            rows.append({
                "provider": "Anthropic",
                "model_name": anthropic_model_name.strip(),
                "raw_output": raw_anthropic,
                "predicted_price": pred_anthropic
            })
        except Exception as e:
            rows.append({
                "provider": "Anthropic",
                "model_name": anthropic_model_name.strip(),
                "raw_output": f"ERROR: {e}",
                "predicted_price": None
            })

    else:
        raise ValueError("Invalid mode selected.")

    df = pd.DataFrame(rows)

    print(df)

    summary_lines = []
    for _, row in df.iterrows():
        summary_lines.append(
            f"{row['provider']} | {row['model_name']} | "
            f"raw={row['raw_output']} | predicted_price={row['predicted_price']}"
        )

    plot_df = df.dropna(subset=["predicted_price"])
    plot_df["label"] = df["provider"] + "\n" + df["model_name"]
    fig = build_bar_chart(plot_df) if len(plot_df) > 0 else None

    # print(plot_df)
    # print(plot_df.dtypes)

    return "\n".join(summary_lines), df, fig

In [ ]:
# Force dark mode
force_dark_mode = """
function refresh() {
    const url = new URL(window.location.href);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.replace(url.toString());
    }
}
refresh();
"""

In [ ]:
# set the theme
theme = gr.themes.Default(
    primary_hue="yellow",
    neutral_hue="gray",
)

In [ ]:
#Gradio
with gr.Blocks(title="Price Estimator App") as demo:
    gr.Markdown(
        """
# Price Estimator

Choose:
- GPT-4.1-nano
- Claude Haiku 4.5

Run one model or both, and compare the predicted prices in a bar chart.
"""
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=420):
            summary_input = gr.Textbox(
                label="Product Summary",
                lines=8,
                placeholder="Example: Apple iPhone 14 128GB unlocked, good battery health, minor scratches"
            )

            run_mode_input = gr.Radio(
                choices=["OpenAI only", "Anthropic only", "Both"],
                value="Both",
                label="Run Mode"
            )

            openai_model_input = gr.Dropdown(
                choices=OPENAI_MODELS,
                value="gpt-4.1-nano",
                label="OpenAI Model"
            )

            anthropic_model_input = gr.Dropdown(
                choices=list(ANTHROPIC_MODEL_OPTIONS.keys()),
                value="Claude Haiku 4.5 (cheaper)",
                label="Anthropic Model"
            )

            fewshot_input = gr.Textbox(
                label="Few-shot examples for Anthropic (optional)",
                lines=6,
                placeholder=(
                    "Format:\n"
                    "iPhone 13 128GB good condition || 480.00\n"
                    "Samsung Galaxy S22 used minor scratches || 420.00"
                )
            )

            run_button = gr.Button("Estimate Price", variant="primary")

        with gr.Column(scale=1, min_width=420):
            summary_output = gr.Textbox(
                label="Output Summary",
                lines=4
            )

            table_output = gr.Dataframe(
                label="Prediction Results",
                #height=220
            )

            chart_output = gr.Plot(
                label="Price Comparison Chart",
                #height=320
            )
            # chart_output = gr.BarPlot(
            #     x="label",
            #     y="predicted_price",
            #     title="Predicted Price by Model",
            #     x_title="Model",
            #     y_title="Predicted Price"
            # )

    run_button.click(
        fn=run_price_estimation,
        inputs=[
            summary_input,
            run_mode_input,
            openai_model_input,
            anthropic_model_input,
            fewshot_input
        ],
        outputs=[
            summary_output,
            table_output,
            chart_output
        ]
    )

if __name__ == "__main__":
    demo.launch(theme=theme, js=force_dark_mode, debug=True, inbrowser=True, show_error=True)